# Sensitivity Analysis Notebook
This notebook evaluates how mortality pricing reacts to parameters such as interest rates and mortality rate shocks, using configuration defaults from `config.py` and the modular `engine/pricing.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import config
from engine.pricing import calculate_pricing

df = pd.read_csv('../data/morality_table.csv')

def get_premium(age, gender, term, sum_assured, interest_rate, shock=1.0):
    # Note: We represent mortality shock as an improvement rate equal to (1 - shock)
    # e.g., if shock is 1.2 (20% increase in qx), improvement rate = 1 - 1.2 = -0.2 (negative improvement)
    improvement_rate = 1.0 - shock
    
    res = calculate_pricing(
        age=age,
        gender=gender,
        term=term,
        sum_assured=sum_assured,
        interest_rate=interest_rate,
        improvement_rate=improvement_rate,
        df_mort=df,
        gender_factors=config.GENDER_FACTORS
    )
    return res['lap_term']

In [ ]:
# Interest Rate Sensitivity
rates = np.linspace(1.0, 12.0, 15)
premiums = [get_premium(config.DEFAULT_AGE, 'Male', config.DEFAULT_TERM, config.SUM_ASSURED, r) for r in rates]

plt.figure(figsize=(10, 5))
plt.plot(rates, premiums, 'o-', color='purple')
plt.title('Premium Sensitivity to Discount / Interest Rates')
plt.xlabel('Interest Rate (%)')
plt.ylabel('Annual Premium ($)')
plt.grid(True)
plt.show()

In [ ]:
# Mortality Shock Sensitivity
shocks = np.linspace(0.8, 1.5, 8)
shk_premiums = [get_premium(config.DEFAULT_AGE, 'Male', config.DEFAULT_TERM, config.SUM_ASSURED, config.INTEREST_RATE * 100, shock=s) for s in shocks]

plt.figure(figsize=(10, 5))
plt.plot(shocks, shk_premiums, 's-', color='red')
plt.title('Premium Sensitivity to Mortality Shocks (Multiplier)')
plt.xlabel('Mortality Multiplier')
plt.ylabel('Annual Premium ($)')
plt.grid(True)
plt.show()